In [ ]:
from pathlib import Path
import sys

sys.path.insert(0, str(Path('code').resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ecg import (
    RECORDS_FOLDER,
    Subject,
    WindowManager,
    align_peaks,
    detect_qrs_for_subjects,
    extract_average_energy,
    get_available_patients,
    initialize_templates,
    load_all_subjects,
    load_patient_table,
    read_subject_data,
    synchronize_peaks,
    update_templates,
)
from ecg.features import split_annotations_by_type

## MIT-BIH Arrhythmia Database

### Arrhythmia Database contains 48 half-hour excerpts of two-channel ambulatory ECG recordings


### Informações dos pacientes


In [ ]:
df = load_patient_table(RECORDS_FOLDER)
df.head()

### Leitura dos dados


##### Data sample


In [ ]:
available_patients = get_available_patients(RECORDS_FOLDER)

subject_id = available_patients[2]
signal, fs, annotation = read_subject_data(subject_id, verbose=True)

signal_channel_1 = signal[:, 0]
signal_channel_2 = signal[:, 1]

In [ ]:
print(annotation.sample[:10])
print(annotation.symbol[:10])
print(set(annotation.symbol))

###### Getting all subjects data


In [ ]:
subjects: list[Subject] = load_all_subjects(RECORDS_FOLDER)
len(subjects)

#### Adaptive Threshold


In [ ]:
detected_indices = detect_qrs_for_subjects(subjects)
print(len(detected_indices))
print(len(detected_indices[0]))

In [ ]:
target_subject = subjects[0]
target_indices = detected_indices[0]

def plot_pt_steps(raw_signal, filtered_signal, derivated_signal, squared_signal, integrated_signal, fs=360, time=5):
    num_samples = time * fs
    time_axis = np.arange(num_samples) / fs

    plt.figure(1, figsize=(10, 4))
    plt.plot(time_axis, raw_signal[:num_samples], color='black')
    plt.title('Sinal Original')
    plt.ylabel('Amplitude')
    plt.xlabel('Tempo (s)')
    plt.tight_layout()

    plt.figure(2, figsize=(10, 4))
    plt.plot(time_axis, filtered_signal[:num_samples], color='blue')
    plt.title('Filtro Passa-Faixa')
    plt.ylabel('Amplitude')
    plt.xlabel('Tempo (s)')
    plt.tight_layout()

    plt.figure(3, figsize=(10, 4))
    plt.plot(time_axis, derivated_signal[:num_samples], color='green')
    plt.title('Derivação Causal')
    plt.ylabel('Amplitude')
    plt.xlabel('Tempo (s)')
    plt.tight_layout()

    plt.figure(4, figsize=(10, 4))
    plt.plot(time_axis, squared_signal[:num_samples], color='orange')
    plt.title('Transformação Não-Linear (Quadrado)')
    plt.ylabel('Amplitude')
    plt.xlabel('Tempo (s)')
    plt.tight_layout()

    plt.figure(5, figsize=(10, 4))
    plt.plot(time_axis, integrated_signal[:num_samples], color='red')
    plt.title('Integração por Janela Móvel')
    plt.ylabel('Amplitude')
    plt.xlabel('Tempo (s)')
    plt.tight_layout()
    plt.show()

def plotar_marcacoes(sinal, indices):
    plt.figure(figsize=(12, 4))
    plt.plot(sinal, color='#1f77b4', label='Sinal original', linewidth=1)

    indices_validos = [i for i in indices if i < len(sinal)]
    amplitudes = [sinal[i] for i in indices_validos]

    plt.scatter(indices_validos, amplitudes, color='red', marker='x', s=50, label='Marcações', zorder=3)
    plt.xlabel('Índice da Amostra')
    plt.ylabel('Amplitude')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

plotar_marcacoes(target_subject.integrated_signal[:1000, 0], target_indices[:3])

##### Post-Processing


In [ ]:
real_peaks = align_peaks(target_indices, target_subject.raw_signal[:, 0], search_window_ms=25, fs=target_subject.fs)
print(target_subject.annotations.sample[1:], target_subject.annotations.sample[1:].shape)
print(real_peaks, real_peaks.shape)

In [ ]:
plotar_marcacoes(target_subject.raw_signal[:1000, 0], real_peaks[:3])

Alguns picos naturalmente não vão ser detectados pelo algoritmo de Pan-Tompkins. Por isso, é necessário fazer a distinção de quais picos foram realmente detectados e quais não foram.

Para isso colocamos um período de 150ms entre um pico real e um pico detectado. Se ele não estiver neste intervalo podemos considerá-lo um falso positivo.


In [ ]:
synced_r_peaks_indices, synced_integrated_peaks_indices, synced_labels = synchronize_peaks(
    annotated_peaks=target_subject.annotations.sample,
    labels=target_subject.annotations.symbol,
    detected_r_peaks=real_peaks,
    detected_int_peaks=target_indices,
    fs=target_subject.fs,
)
len(synced_r_peaks_indices), len(synced_labels)

Windowing


In [ ]:
window_manager = WindowManager(
    synced_filtered_peaks=synced_r_peaks_indices,
    synced_labels=synced_labels,
    synced_integrated_peaks=synced_integrated_peaks_indices,
    integrated_signal=target_subject.integrated_signal[:, 0],
    filtered_signal=target_subject.filtered_signal[:, 0],
    window_span_ms=400,
    fs=target_subject.fs,
)
len(window_manager.filtered_r_peaks_windows)

Feature extraction


In [ ]:
templates = initialize_templates(window_manager.filtered_r_peaks_windows)
templates[0].shape

### Distribuições


In [ ]:
valid_ages = df['Idade'][df['Idade'] != -1]

plt.figure(figsize=(8, 6))
plt.hist(valid_ages, edgecolor='black')
plt.xlabel('Idade')
plt.ylabel('Frequência')
plt.title('Distribuição das idades')
plt.grid(linestyle='--', axis='y')
plt.show()

In [ ]:
gender_count = df['Gênero'].value_counts()

plt.figure(figsize=(6, 4))
plt.bar(
    gender_count.index,
    gender_count.values,
    edgecolor='black',
    color=['lightblue', 'salmon'],
    width=0.6,
)
plt.xlabel('Gênero')
plt.ylabel('Frequência')
plt.title('Distribuição de sexos')
plt.show()

In [ ]:
normal_times, arrhythmia_times = split_annotations_by_type(annotation, fs)
normal_count = len(normal_times)
arrythm_count = len(arrhythmia_times)
print(normal_count, arrythm_count)

labels = ['Ritmo Normal', 'Arritmias']
values = [normal_count, arrythm_count]
bars = plt.bar(labels, values, color=['green', 'red'], width=0.25)
plt.ylabel('Quantidade')
plt.title('Anotações por tipo')
plt.grid(axis='y', linestyle='--', alpha=0.7)
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2, height, str(height),
             ha='center', va='bottom', fontsize=10, fontweight='bold')
plt.show()

In [ ]:
normal_count / (normal_count + arrythm_count) * 100

### Feature extraction


In [ ]:
extract_average_energy(window_manager.filtered_r_peaks_windows[0])

### Subamostragem
